# 09 — Sumarização por Grafos e Avaliação

**Disciplina 06 — Processamento de Linguagem Natural**  |  **Capítulo 09/17**

> Biblioteca pessoal de Ciência de Dados · capítulo didático independente · núcleo executável sem internet


### Materiais complementares

- [Guia técnico EP — Sumarização com Grafos e Avaliação](./reference_materials/09_sumarizacao_grafos_avaliacao.pdf)
- [Cheat sheet cookbook — Sumarização com Grafos e Avaliação](./cheat_sheets/cheat_sheet_sumarizacao_grafos_avaliacao.pdf)
- [Documentação e saídas](./README.pt-BR.md)


## Objetivos de aprendizagem

- Construir um grafo de similaridade entre frases.
- Implementar um ranking inspirado em TextRank.
- Conhecer ROUGE e limites da avaliação automática.

## Rastreabilidade e pré-requisitos

- **Origem e referência:** Aulas 02–04 + aprofundamento.
- **Conhecimentos reutilizados:** 05 — validação; 03 — estatística.
- **Progressão:** novos conceitos são aprofundados aqui sem reconstruir capítulos anteriores.

## Fundamentos conceituais

Na sumarização por grafos, cada frase é um nó e a similaridade entre frases define as arestas. Frases conectadas a outras consideradas centrais podem receber score elevado. Isso não implica que selecioná-las produza um resumo sem redundância. Métodos como ROUGE avaliam sobreposição lexical com resumos de referência, mas não garantem fidelidade factual.

## Modelo matemático e interpretação

No PageRank, uma atualização simplificada combina teletransporte e massa recebida dos vizinhos: r_{t+1}=α Pᵀr_t+(1−α)u. A matriz P é estocástica após normalizar suas linhas.

## Estudo guiado

**Pergunta de investigação:** como demonstrar o conceito central com um exemplo pequeno, auditável e reproduzível?

Nas células seguintes, inspecione as entradas, execute os procedimentos e explique com suas próprias palavras o resultado. Ao trocar o texto ou o parâmetro, observe se o comportamento permanece coerente. Esses resultados são ilustrações pedagógicas, não benchmarks.

## Implementação reproduzível

O código abaixo usa textos sintéticos ou bibliotecas locais. **Não executa downloads nem acessa APIs externas.** Exemplos opcionais só rodam quando a dependência já está disponível; a ausência deles não interrompe o notebook.

In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
frases=['A biblioteca recebeu livros novos.','Os estudantes leram os livros na biblioteca.','A universidade ampliou os horários de estudo.','Novos livros foram catalogados ontem.']
M=TfidfVectorizer().fit_transform(frases)
W=cosine_similarity(M);np.fill_diagonal(W,0)
row=W.sum(axis=1,keepdims=True)
P=np.divide(W,row,out=np.full_like(W,1/len(frases)),where=row>0)
r=np.ones(len(frases))/len(frases)
for _ in range(60):r=.85*(P.T@r)+.15/len(frases)
print('Ranking semelhante ao TextRank:',sorted(zip(np.round(r,3),frases),reverse=True))

Ranking semelhante ao TextRank: [(np.float64(0.317), 'A biblioteca recebeu livros novos.'), (np.float64(0.315), 'Os estudantes leram os livros na biblioteca.'), (np.float64(0.228), 'Novos livros foram catalogados ontem.'), (np.float64(0.141), 'A universidade ampliou os horários de estudo.')]


**Interpretação guiada:** verifique se o resultado corresponde à fórmula e à hipótese discutidas. Anote o que muda com a entrada e quais conclusões **não** podem ser generalizadas a partir deste exemplo.

In [2]:
def rouge1_recall(referencia, candidato):
    from collections import Counter
    import re
    words=lambda s:Counter(re.findall(r'\b\w+\b',s.lower()))
    a,b=words(referencia),words(candidato)
    return sum((a&b).values())/max(sum(a.values()),1)
print('ROUGE-1 recall (simplificado):',rouge1_recall('a biblioteca recebeu novos livros','a biblioteca recebeu livros'))

ROUGE-1 recall (simplificado): 0.8


**Interpretação guiada:** verifique se o resultado corresponde à fórmula e à hipótese discutidas. Anote o que muda com a entrada e quais conclusões **não** podem ser generalizadas a partir deste exemplo.

## Exercícios orientados

1. **Compreensão:** explique qual informação o método preserva e qual descarta. Relacione a resposta com os objetivos do capítulo.
2. **Experimento:** altere uma frase e um parâmetro do código. Registre a diferença e uma hipótese explicativa.
3. **Avaliação crítica:** descreva uma situação em que a técnica geraria uma resposta inadequada. Proponha uma mitigação.
4. **Transferência:** conecte o conceito a uma disciplina anterior e identifique o que não precisa ser revisto.

**Resolução comentada (diretriz de resposta):** o método representa somente um recorte do fenômeno linguístico; mudanças de entrada alteram a saída segundo a regra estudada. Uma boa análise identifica perda de informação, contexto e risco de generalização. Para a resposta específica, justifique com os valores efetivamente obtidos na execução acima.

## Erros comuns, limitações e boas práticas

Limiar de similaridade, sentenças isoladas e remoção de redundância alteram o resultado. Não confundir ROUGE alto com resumo correto.

- Separe demonstração de produção: datasets sintéticos não demonstram generalização.
- Registre versões das dependências e parâmetros; não inclua credenciais no notebook.
- Consulte a documentação oficial antes de adaptar códigos históricos de 2021.

## Consulta rápida

| Item | Resumo |
|---|---|
| Conceito | Sumarização por Grafos e Avaliação |
| Fonte histórica | Aulas 02–04 + aprofundamento |
| Dependências do exemplo | Python, NumPy / scikit-learn / NLTK / spaCy conforme as células |
| Critério de sucesso | Interpretar as saídas e explicar limites |
| Próximo passo | Aplicar a técnica a outro corpus de demonstração |

## Aprofundamento v2 — auditoria e aplicação

**Ponto de atenção deste capítulo:** redundância de sumários. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **redundância de sumários** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **redundância de sumários** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.
